In [0]:
import random
import json
from datetime import datetime as dt, timedelta

In [0]:
# Block 1 config 

CYCLES_PER_SHIFT = 35
PAYLOAD_TARGETS = {"777": 90, "785": 140}
PAYLOAD_VARIATION = 0.10
TELEMETRY_PATH = "/Volumes/mining/landing/telemetry"
GROUND_TRUTH_PATH = "/Volumes/mining/ground_truth/truth"
NUM_SHIFTS = 5          # generate 5 consecutive shift-days
SHIFT_BASE_DATE = dt(2026, 8, 1)   # first shift's date; each shift = +1 day


In [0]:
print(PAYLOAD_TARGETS)

In [0]:
# Block 2 The fleet
trucks = [
    {"truck_id": "T01", "model": "777", "capacity": 90},
    {"truck_id": "T02", "model": "777", "capacity": 90},
    {"truck_id": "T03", "model": "777", "capacity": 90},
    {"truck_id": "T04", "model": "777", "capacity": 90},
    {"truck_id": "T05", "model": "785", "capacity": 140},
    {"truck_id": "T06", "model": "785", "capacity": 140},
    {"truck_id": "T07", "model": "785", "capacity": 140},
    {"truck_id": "T08", "model": "785", "capacity": 140},
]

for truck in trucks:
    print(f"Truck ID: {truck['truck_id']}, Model: {truck['model']}, Capacity: {truck['capacity']} tons")


In [0]:
# Block 3 — Generate NUM_SHIFTS shifts of haul cycles.
# Each shift is one simulated day with its own SHIFT_ID and file.
# v1: NO corruption yet — reported_tonnage == true_tonnage.

CYCLE_MINUTES = 15

all_shifts = []  # list of (shift_id, telemetry_records, truth_records)

for shift_num in range(NUM_SHIFTS):
    shift_date = SHIFT_BASE_DATE + timedelta(days=shift_num)
    SHIFT_ID = shift_date.strftime("%Y%m%d")   # e.g. 20260801, unique per day
    shift_start = shift_date.replace(hour=6)    # shift starts 06:00

    telemetry_records = []
    truth_records = []

    for truck in trucks:
        for cycle_number in range(CYCLES_PER_SHIFT):
            target = truck["capacity"]
            low  = target * (1 - PAYLOAD_VARIATION)
            high = target * (1 + PAYLOAD_VARIATION)
            true_tonnage = round(random.uniform(low, high), 1)

            cycle_id = f"{SHIFT_ID}-{truck['truck_id']}-C{cycle_number:02d}"
            cycle_time = shift_start + timedelta(minutes=cycle_number * CYCLE_MINUTES)
            timestamp = cycle_time.isoformat()

            telemetry_records.append({
                "cycle_id": cycle_id,
                "shift_id": SHIFT_ID,
                "truck_id": truck["truck_id"],
                "model": truck["model"],
                "timestamp": timestamp,
                "reported_tonnage": true_tonnage,
            })

            truth_records.append({
                "cycle_id": cycle_id,
                "shift_id": SHIFT_ID,
                "truck_id": truck["truck_id"],
                "true_tonnage": true_tonnage,
            })

    all_shifts.append((SHIFT_ID, telemetry_records, truth_records))

total = sum(len(t) for _, t, _ in all_shifts)
print(f"Generated {NUM_SHIFTS} shifts, {total} telemetry records total")

In [0]:
# Block 4 — Write telemetry as NDJSON, one file per shift.
for shift_id, telemetry_records, truth_records in all_shifts:
    telemetry_file = f"{TELEMETRY_PATH}/telemetry_{shift_id}.json"
    with open(telemetry_file, "w") as f:
        for record in telemetry_records:
            f.write(json.dumps(record) + "\n")
    print(f"Wrote {len(telemetry_records)} telemetry records to {telemetry_file}")

In [0]:
# Block 5 — Write answer key, one file per shift, to walled-off ground_truth.
for shift_id, telemetry_records, truth_records in all_shifts:
    truth_file = f"{GROUND_TRUTH_PATH}/truth_{shift_id}.json"
    with open(truth_file, "w") as f:
        for record in truth_records:
            f.write(json.dumps(record) + "\n")
    print(f"Wrote {len(truth_records)} truth records to {truth_file}")

In [0]:
# Block 6 Peek at the first 3 telemetry lines to confirm shape.
with open(telemetry_file) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(line.strip())

In [0]:
dbutils.fs.ls("/Volumes/mining/landing/telemetry/")
dbutils.fs.ls("/Volumes/mining/ground_truth/truth/")

In [0]:
# Delete FILES inside each folder — never the folders themselves.
for f in dbutils.fs.ls("/Volumes/mining/landing/telemetry/"):
    dbutils.fs.rm(f.path)

for f in dbutils.fs.ls("/Volumes/mining/ground_truth/truth/"):
    dbutils.fs.rm(f.path)

for f in dbutils.fs.ls("/Volumes/mining/bronze/checkpoints/"):
    dbutils.fs.rm(f.path, recurse=True)

print("Files cleared.")

In [0]:
spark.sql("DROP TABLE IF EXISTS mining.bronze.haul_events")
spark.sql("DROP TABLE IF EXISTS mining.silver.haul_events")
spark.sql("DROP TABLE IF EXISTS mining.silver.quarantine")
spark.sql("DROP TABLE IF EXISTS mining.gold.production_reconciliation")
print("Tables dropped.")

In [0]:
dbutils.fs.ls("/Volumes/mining/landing/telemetry/")